# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [4]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('GEMINI_API_KEY')

if api_key and api_key.startswith('AQ.') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gemini-3.5-flash'
gemini = OpenAI(
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    api_key=api_key,
)

API key looks good so far


In [5]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https:/

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [6]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [7]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [8]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17

In [9]:
def select_relevant_links(url):
    response = gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [10]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'curriculum page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'company page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'linkedin profile',
   'url': 'https://www.linkedin.com/in/eddonner/'}]}

In [11]:
select_relevant_links("https://home.cegsoft.com/")

{'links': [{'type': 'about page', 'url': 'https://home.cegsoft.com/about'},
  {'type': 'careers page', 'url': 'https://home.cegsoft.com/career'}]}

In [13]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [14]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gemini-3.5-flash
Found 6 relevant links


{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'resume page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'company page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'linkedin profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'blog page', 'url': 'https://edwarddonner.com/posts/'}]}

In [15]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-3.5-flash
Found 6 relevant links


{'links': [{'type': 'careers page',
   'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'brand resources', 'url': 'https://huggingface.co/brand'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'linkedin profile',
   'url': 'https://www.linkedin.com/company/huggingface/'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [16]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [18]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gemini-3.5-flash
Found 8 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Hardware
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
autotrust/JEV-27B-VL
Updated
about 10 hours ago
•
1.53M
•
1.98k
Cloudflare/clef
Updated
about 10 hours ago
•
9.51k
•
1.82k
autotrust/GEV-26B-Decide
Updated
5 days ago
•
896k
•
1.32k
google/embeddinggemma-2
Updated
1 day ago
•
7.56k
•
953
Aleph-Alpha/Kolibri-1
Updated
5 days ago
•

In [19]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [20]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [21]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-3.5-flash
Found 9 relevant links


"\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nHardware\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nautotrust/JEV-27B-VL\nUpdated\nabout 10 hours ago\n•\n1.53M\n•\n1.98k\nCloudflare/clef\nUpdated\nabout 10 hours ago\

In [32]:
def create_brochure(company_name, url):
    response = gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    return result
    #display(Markdown(result))

In [33]:
def create_brochure_spanish(content):
    response = gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "Translate whatever comes here to spanish"},
            {"role": "user", "content": content}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [34]:
create_brochure_spanish(create_brochure("HuggingFace", "https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gemini-3.5-flash
Found 9 relevant links


# Hugging Face: La comunidad de IA que construye el futuro

Hugging Face es la plataforma de colaboración líder en el mundo para el aprendizaje automático (*machine learning*). Sirve como el centro neurálgico donde la comunidad global de IA se reúne para crear, compartir y colaborar en modelos, conjuntos de datos (*datasets*) y aplicaciones. 

Ya sea que represente a una empresa que busca escalar sus capacidades de IA, a un inversor que busca el centro de gravedad de la tecnología moderna, o a un creador que desea generar un impacto global, Hugging Face es el lugar donde se está diseñando el futuro de la IA.

---

## El hogar del aprendizaje automático

En su esencia, Hugging Face es la plataforma de colaboración definitiva para el desarrollo de aprendizaje automático. Ofrece un ecosistema abierto y masivo que impulsa la inteligencia artificial moderna:

*   **Más de 2 millones de modelos:** Desde modelos de lenguaje grande (LLM) de vanguardia hasta visión por computadora y modelos multimodales, alojados y listos para implementar.
*   **Más de 500,000 conjuntos de datos:** Conjuntos de datos diversos y de alta calidad que abarcan desde texto conversacional hasta video, lo que permite un entrenamiento de modelos robusto.
*   **Más de 1 millón de aplicaciones (Spaces):** Aplicaciones y demostraciones de IA interactivas y en vivo que se ejecutan directamente en la plataforma, mostrando los últimos avances en tiempo real.

---

## Para nuestros clientes: IA de nivel empresarial

Hugging Face ofrece herramientas e infraestructura robustas para ayudar a las empresas a realizar la transición de la investigación a la producción de manera segura y eficiente.

### Soluciones para empresas y equipos
Ayudamos a las organizaciones a desarrollar IA de forma segura con planes y herramientas especializados, diseñados para equipos de desarrollo profesional:
*   **Hugging Face PRO y Soporte Empresarial:** Obtenga acceso directo a expertos en IA para acelerar sus ciclos de desarrollo y resolver obstáculos técnicos.
*   **Puntos de conexión de inferencia (Inference Endpoints):** Implemente cualquier modelo en una infraestructura segura, con escalado automático y altamente optimizada con solo unos pocos clics.
*   **Contenedores de almacenamiento (Storage Buckets):** Almacene y gestione activos de datos a gran escala y pesos de modelos de forma fluida.
*   **Proveedores de inferencia:** Integraciones profundas con los principales proveedores de nube y hardware para ejecutar modelos donde residen sus datos.

---

## Para inversores: El epicentro de la revolución de la IA

Hugging Face se ha consolidado como la capa indispensable de la pila tecnológica (*stack*) de IA moderna.

*   **Efectos de red incomparables:** Al albergar millones de modelos y conjuntos de datos de código abierto, Hugging Face se ha convertido en el punto de partida predeterminado para desarrolladores, investigadores y empresas emergentes (*startups*) en todo el mundo.
*   **Alianzas en toda la industria:** Colaboramos con los nombres más importantes de la tecnología —incluidos Google, Cloudflare y los principales fabricantes de chips— para garantizar que nuestro ecosistema siga siendo el estándar de la industria.
*   **Embudo empresarial de alto valor:** Nuestra enorme comunidad de desarrolladores se canaliza de forma natural hacia nuestros Enterprise Hubs de pago, cuentas PRO y servicios de inferencia dedicados, lo que convierte a Hugging Face en un motor de negocio altamente escalable.

---

## Carreras: Únase al movimiento de código abierto

En Hugging Face, creemos que el futuro de la IA debe ser abierto, colaborativo y accesible para todos.

### Nuestra cultura
Fomentamos un entorno de colaboración de código abierto, aprendizaje continuo y gran autonomía. Valoramos a los creadores apasionados por democratizar la IA, contribuir a la tecnología impulsada por la comunidad y resolver desafíos de ingeniería complejos.

### Vacantes actuales
Estamos buscando activamente personas talentosas que nos ayuden a construir la próxima generación de herramientas de IA. Si es ingeniero de aprendizaje automático, desarrollador de software, *developer advocate* o experto en soluciones empresariales y desea dar forma al futuro de la tecnología, consulte nuestras vacantes actuales en nuestra página de Carreras y únase al equipo que está capacitando a millones de creadores en todo el mundo.

In [24]:
create_brochure("CEGSoft", "https://home.cegsoft.com/")

Selecting relevant links for https://home.cegsoft.com/ by calling gemini-3.5-flash
Found 3 relevant links


# CEGSoft: Puerto Rico’s Leading Software & Technology Partner

Welcome to CEGSoft, the premier technology company in Puerto Rico. For over 25 years, we have been at the forefront of digital transformation, creating simple, secure, and collaborative cloud-based software solutions that optimize workflows and drive efficiency. 

Whether you are a prospective customer seeking to streamline your operations, an investor looking to back a proven market leader, or a talented professional aiming to build a rewarding career, CEGSoft is your gateway to the future of technology.

---

## What We Do: Your Digital Office, Simplified

Our mission is simple: we design the software, and you experience the ease of a highly organized, productive, and secure digital environment. Our cloud-based applications feature top-tier security, allowing our users to work seamlessly from anywhere in the world.

### Our Core Products

*   **Taxmania:** A fast, secure, and user-friendly web application certified by the Puerto Rico Department of the Treasury (Hacienda) and the IRS, designed for individuals to prepare and file their tax returns online.
*   **Expert Tax:** The leading tax preparation and electronic filing platform designed specifically for accounting and tax professionals in Puerto Rico.
*   **Edi:** An intuitive digital document management and archiving program that allows organizations of all sizes to securely store, organize, and structure their digital files.
*   **Followit:** A comprehensive case management platform engineered for large corporations and government agencies to eliminate incomplete tasks, track deadlines, and manage workflows.

---

## Why Investors and Customers Trust Us

CEGSoft has established itself as an indispensable technological cornerstone for businesses and individuals alike. Our proven track record speaks for itself:

*   **25+ Years of Experience:** Over two decades of expertise in developing and deploying mission-critical software.
*   **150,000+ Active Users:** Trusted by individuals and professionals across the region.
*   **1,500+ Commercial Clients:** We provide secure software solutions trusted by industry leaders in accounting, banking, education, and government.
*   **Market Leadership:** Our certified tax software platforms handle a major share of electronic tax filings in Puerto Rico, demonstrating robust software scalability and compliance with strict governmental standards.

---

## Life at CEGSoft: Culture and Careers

Are you ready to build the future of tech? As "La mejor compañía de tecnología en Puerto Rico" (The best technology company in Puerto Rico), we pride ourselves on fostering a collaborative, innovative, and customer-first work environment.

### Why Join CEGSoft?

*   **A Culture of Purpose:** At CEGSoft, we design with the end-user in mind. Every line of code our team writes directly relieves stress and boosts productivity for thousands of users.
*   **Modern Work Environment:** Our cloud-first approach means we embrace flexible, modern working styles that empower our team to work securely and efficiently.
*   **Innovation & Growth:** With a diverse portfolio of B2C and B2B products, our developers, product managers, and customer success specialists continuously solve challenging problems and grow their professional skill sets.
*   **Local Roots, Big Impact:** Be part of a proud Puerto Rican enterprise that makes a massive daily impact on the island's economic infrastructure.

---

## Connect With Us

Discover how CEGSoft can transform your business, your investment portfolio, or your career.

*   **Website:** Contact us today to learn more about our enterprise solutions or explore open job opportunities under our **Empleos** (Careers) section.
*   **Transform Your Digital Office:** Explore our secure, cloud-based applications and join over 150,000 satisfied users.

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [25]:
def stream_brochure(company_name, url):
    stream = gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [ ]:


stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-3.5-flash
Found 10 relevant links


# Hugging Face: The AI Community Building the Future

Hugging Face is the world's leading collaboration platform for machine learning. It serves as the central hub where the global AI community—ranging from independent researchers to enterprise teams—comes together to build, explore, and run the future of artificial intelligence. 

Whether you are an enterprise customer looking to scale your AI operations, an investor seeking the heart of the modern tech stack, or a recruit wanting to build state-of-the-art open-source technology, Hugging Face is where the future of AI is happening.

---

## The Home of Machine Learning

At its core, Hugging Face is the definitive platform for collaborative machine learning development. The platform hosts an unprecedented volume of open-source AI assets:

*   **2 Million+ Models:** Access, test, and implement cutting-edge models for natural language processing, computer vision, audio, and multimodal tasks (including trending models from Google, Cloudflare, Aleph Alpha, and more).
*   **500,000+ Datasets:** Fuel your AI training and evaluation pipelines with high-quality, diverse datasets spanning dozens of languages and industries.
*   **1 Million+ Applications (Spaces):** Build and deploy interactive machine learning applications in minutes. Developers use Spaces to showcase demos, run benchmarks, and deploy tools like HuggingChat, utilizing cutting-edge backend hardware like ZeroGPU.

---

## Solutions for Enterprise & Customers

Hugging Face provides robust infrastructure, security, and dedicated support for organizations looking to integrate machine learning into their production systems safely and efficiently.

### Enterprise Support & Scale
We help teams move from research to production seamlessly. Our enterprise suite includes:
*   **Team & Enterprise Plans:** Private collaboration spaces with secure organization management.
*   **Inference Endpoints:** Deploy any model on secure, auto-scaling, and highly available cloud infrastructure with just a few clicks.
*   **Storage Buckets:** Safe, centralized storage optimized for massive machine learning datasets and model weights.
*   **Hugging Face PRO & Enterprise Support:** Direct access to Hugging Face’s team of world-class ML engineers to accelerate your development cycles.

---

## Culture & Careers: Why Join Hugging Face?

### Our Culture
At Hugging Face, we believe that open-source and collaborative development lead to safer, better, and more equitable AI technology. Our culture is characterized by:
*   **Openness and Collaboration:** We host open-source projects, publish research, and participate daily in community building through our forums, Discord, and GitHub.
*   **Curiosity-Driven:** We keep our finger on the pulse of innovation, hosting tools like "Daily Papers" to keep our team and community at the cutting edge of scientific discovery.
*   **Impact:** Our employees build the tools that power AI at thousands of companies worldwide, shaping how the world interacts with artificial intelligence.

### Career Opportunities
As a rapidly growing hub for AI, we are constantly seeking passionate builders, researchers, and engineers. Working at Hugging Face means collaborating with the brightest minds in deep learning, software engineering, and community building to build tools that democratize AI for everyone.

---

## Join the AI Revolution

Hugging Face is not just a platform; it is a movement. Join millions of developers, researchers, and businesses who are creating, discovering, and collaborating on machine learning today.

*   **Explore:** huggingface.co
*   **Connect:** Join our Discord, Forums, or follow our GitHub to become part of the community.

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>